# Football Matches Statistics

## 1. Initial Exploration and Data Cleaning

The purpose of this notebook is to explore and clean the football dataset before performing the Exploratory Data Analysis (EDA).

This notebook follows the standard workflow:


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

: 

## 1.1 Load the dataset

In [ ]:
df = pd.read_csv("dataset/football_matches_dataset.csv")

## 1.2. Initial Data Exploration

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
list(df.columns)

In [ ]:
[col for col in df.columns if "id" in col.lower()]

In [ ]:
df["Unnamed: 0"].head()

In [ ]:
df["Unnamed: 0"].duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include="all")

## 1.3. Missing Values

We checked for missing values to identify incomplete records and determine whether they should be removed, filled, or kept for further analysis.

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sort_values(ascending=False) #ordering in a DESC way

In [ ]:
missing = df.isnull().sum()     #check

missing_percentage = (missing / len(df)) * 100

pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percentage.round(2)
}).sort_values("Missing Values", ascending=False)

Findings of isnull:

Several columns related to previous team performance contain missing values.

These missing values are expected because, at the beginning of each season, teams have not yet played enough matches to calculate previous statistics.

At this stage, no rows will be removed until we evaluate whether these missing values affect our research questions.

## 1.4. Duplicate Rows

We checked whether the dataset contains duplicated rows that could affect the analysis.

In [ ]:
df.duplicated().sum()

Where not found any duplicated rows "0", so, no duplicate removal is required.

## 1.5. Index Column Review

The dataset contains an additional column called `Unnamed: 0`.

This column appears to be an automatically generated index from a previous CSV export rather than a meaningful identifier.

We verified that it contains unique values, but it will not be used in the analysis.

In [ ]:
df["Unnamed: 0"].duplicated().sum()

This column will be removed during the data cleaning process because it does not provide analytical value.

## 1.6. Unique Values

We checked the number of unique values in each column to better understand which variables are categorical, numerical, or identifiers.

In [ ]:
unique_values = pd.DataFrame({
    "Unique Values": df.nunique()
}).sort_values("Unique Values", ascending=False)

unique_values

Findings of Unique Values

The unique values check helps identify categorical variables such as `league`, `season`, and team formations, as well as high-cardinality variables such as `home_team`, `away_team`, and `datetime`.

The `Unnamed: 0` column appears to be an automatically generated index from a previous CSV export and does not provide analytical value. Therefore, it will be removed during the data cleaning process.

## 1.7. Data Types

We reviewed the data types of each column to identify variables that may require conversion.

In [ ]:
df.dtypes

In [ ]:
df.info()

Findings:

Most variables have the expected data types.

However, the `date` and `datetime` columns should be reviewed to determine which one will be used in the final dataset.

## 1.8. Check Invalid Values

In [ ]:
(df["home_score"] < 0).sum()

In [ ]:
(df["home_xg"] < 0).sum()

## 1.9. Check Categorical Values

Leagues

In [ ]:
df["league"].value_counts()

Seasons

In [ ]:
df["season"].value_counts()

Formations

In [ ]:
df["home_formation"].value_counts()

## 1.10. Football-Specific Data Validation

Will check data which cannot be negative for the project

In [ ]:
# Goals cannot be negative
(df["home_score"] < 0).sum()
(df["away_score"] < 0).sum()

In [ ]:
# Expected Goals cannot be negative
(df["home_xg"] < 0).sum()
(df["away_xg"] < 0).sum()

In [ ]:
# Current points should not be negative
(df["home_points"] < 0).sum()
(df["away_points"] < 0).sum()

## Quick Summary

In [ ]:
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Unique Values": df.nunique()
})

summary

# 2. Initial Data Cleaning

### Planned Cleaning Actions

Based on the initial exploration, the following cleaning decisions will be applied:

- Remove the `Unnamed: 0` column because it is an exported index.
- Convert the `datetime` column to datetime format.
- Evaluate whether the `date` column is redundant.
- Keep missing values in previous-performance variables until we determine whether they affect our research questions.
- Create new engineered features for the analysis. (AS EXTRA)

## 2.1. Removing the "Unnamed: 0" column

In [ ]:
clean_df = df.drop(columns=["Unnamed: 0"])

In [ ]:
df.head()

In [ ]:
clean_df.head()

Creating a new DataFrame as **"clean_df"** to storage all our changes from de origina "df"

We passed from **"df"** with 55 columns, to **"clean_df"**(new ) with 54 columns.

## 2.2. Check and compare **"date"** and **"datetime"** columns

In [ ]:
clean_df[["date", "datetime"]].head()

In [ ]:
clean_df[["date", "datetime"]].dtypes

In [ ]:
clean_df["datetime"] = pd.to_datetime(clean_df["datetime"])

With this step we change the type od `datetime` to a real datetime64[ns]

In [ ]:
clean_df[["date", "datetime"]].dtypes

In [ ]:
same_date = (clean_df["date"] == clean_df["datetime"].dt.strftime("%B %d %Y")) #.dt panda serie for dates, 
same_date.all()

Both columns have the same dates


### Cleaning Decision

The dataset contains two columns representing the match date.

- `date` stores the date as a text string.
- `datetime` stores the same information in a format that can be converted into a proper datetime object.

At this stage, both columns will be kept until **THE TEAM** agrees on which one should be used in the final cleaned dataset.

NOTE: i will go by `datetime`, for better uses in the future, where we could access to years (["datetime"].dt.year), months (.dt.month), day (.dt.day_name())... 

# 3. Feature Engineering

In this section, we create new variables from the existing columns to support the analysis of our four research questions.

These new features will help us compare match results, goals, Expected Goals (xG), team form, points, ratings, and formations more clearly.

### 3.1. Match Result

To simplify the analysis, we created a new categorical variable called `match_result`.

Instead of comparing home and away goals repeatedly, this feature directly identifies whether the match ended in a Home Win, Draw, or Away Win.

This feature will be used throughout the EDA and SQL analysis, as it supports all four research questions.

In [ ]:
clean_df.head(1) #exploring the table for getting the columns which will be needed

In [ ]:
conditions = [
    clean_df["home_score"] > clean_df["away_score"],
    clean_df["home_score"] < clean_df["away_score"]
]

choices = [
    "Home Win",
    "Away Win"
]

clean_df["match_result"] = np.select(
    conditions,
    choices,
    default="Draw"
)


Why `np.select()`?

This feature could also be created using a `for` loop, which was my first approach.

However, I decided to use `np.select()` because it applies the conditions to the entire DataFrame at once (vectorized operations), making the code more efficient and more consistent with pandas best practices.

The variables `conditions` and `choices` are not reserved keywords. They are simply descriptive variable names chosen to improve readability. They could be renamed if needed.

The only reserved parameter in this function call is `default`, which defines the value assigned when none of the specified conditions are met.

In [ ]:
clean_df["match_result"].value_counts()

In [ ]:
clean_df[["home_score", "away_score", "match_result"]].head(10)

Findings

The `match_result` feature was successfully created.

The distribution of results shows that Home Wins are the most frequent outcome, followed by Away Wins and Draws. This distribution is consistent with the well-known home advantage observed in professional football.

### 3.2. Goal Difference

In [ ]:
clean_df["goal_difference"] = clean_df["home_score"] - clean_df["away_score"]

In [ ]:
clean_df[["home_score", "away_score", "goal_difference"]].head(5)

Findings

The `goal_difference` feature was successfully created and validated.

Positive values represent home victories, negative values represent away victories, and zero represents draws.

The observed range (-9 to 9) appears realistic for professional football matches, indicating that the feature has been calculated correctly.

Now created `goal_difference` let test some statistics

In [ ]:
clean_df["goal_difference"].describe() 

Some conclusions:

MIN 
Show that largest AWAY victory was by **-9** goals of difference (cause it is negative number)

MAX 
Show that largest HOME victory was by **9** goals of difference (cause it is positive number)

MEDIAN
Value is **0**, meaning that the center of the distribution is a goal difference of zero.
This suggests that most matches are relatively balanced, with many ending in draws or being decided by a small goal margin.

### 3.4. Total Goals

In [ ]:
# Create total_goals
clean_df["total_goals"] = clean_df["home_score"] + clean_df["away_score"]

In [ ]:
clean_df[["home_score", "away_score", "total_goals"]].head()

The `total_goals` feature represents the total number of goals scored in each match.

### 3.5. Total Expected Goals

In [ ]:
# Create total_xg
clean_df["total_xg"] = clean_df["home_xg"] + clean_df["away_xg"]

In [ ]:
clean_df[["home_xg", "away_xg", "total_xg"]].head()

In [ ]:
clean_df[["total_goals","total_xg"]].describe()

The `total_xg` feature represents the combined expected goals generated by both teams.

Both features were successfully validated and will support analyses related to attacking performance and scoring efficiency.

### 3.6. Expected Goal Difference

In [ ]:
clean_df["xg_difference"] = clean_df["home_xg"] - clean_df["away_xg"]

In [ ]:
clean_df[["home_xg", "away_xg", "xg_difference"]].head()

The `xg_difference` feature compares the expected goals generated by both teams before each match.

Positive values indicate that the home team generated more expected goals, while negative values indicate that the away team generated more expected goals.

This feature will help evaluate whether a higher expected goal difference is associated with better match performance.

In [ ]:
clean_df[["xg_difference"]].describe()

### 3.7. Point Difference

The ``points_difference`` feature compares the league points accumulated by both teams before the match.

Positive values indicate that the home team entered the match with more league points, while negative values indicate that the away team had more points.

This feature will help evaluate whether teams with a stronger league position tend to achieve better match performance.

In [ ]:
clean_df["points_difference"] = clean_df["home_points"] - clean_df["away_points"] 

In [ ]:
clean_df[["home_points", "away_points", "points_difference"]].tail()

In [ ]:
clean_df[["points_difference"]].describe()

Findings

The descriptive statistics indicate that the feature was calculated correctly.

Positive values indicate matches where the home team generated more expected goals, while negative values indicate higher expected goals for the away team.

This feature will support analyses comparing expected performance with actual match outcomes.

### 3.8.1 Recent Form Difference

In [ ]:
#newcol = (
 #   clean_df.home_points_last_4_matches
  #  +
   # clean_df.away_points_last_4_matches
#)

reviewing this feature, noticed that the first approach stored the result in `newcol` by **adding** the recent points of both teams:

newcol = (
    clean_df.home_points_last_4_matches -
    clean_df.away_points_last_4_matches
)

even with this calculation is mathematically correct, it represents the **combined recent points of both teams**, rather than comparing which team was in better form before the match.

the question objective is to analyse **recent team form**, and comparing the two teams is more meaningful than summing their points.

For this reason, we decided to calculate the feature as the difference between the home and away teams' recent points:


clean_df["recent_form_difference"] = (clean_df["home_points_last_4_matches"] - clean_df["away_points_last_4_matches"])


this version is easier to interpret:

- Positive values indicate that the home team entered the match in better recent form.
- Negative values indicate that the away team entered the match in better recent form.
- Values close to zero indicate that both teams had a similar recent performance.

I believe this is more close to our research question:

> **Does recent team form influence match outcomes?**

In [ ]:
clean_df["recent_form_difference"] = (clean_df["home_points_last_4_matches"] - clean_df["away_points_last_4_matches"])

### 3.8.2 Recent Form Difference

The `recent_form_difference` feature compares the recent performance of both teams based on the total number of points earned in their previous four matches.

Positive values indicate that the home team entered the match in better recent form, while negative values indicate that the away team had better recent form.

This feature will help evaluate whether recent team form influences match outcomes.

In [ ]:
clean_df["recent_form_difference"] = (clean_df["home_points_last_4_matches"] - clean_df["away_points_last_4_matches"])

In [ ]:
clean_df[["home_points_last_4_matches", "away_points_last_4_matches", "recent_form_difference"]].tail()

In [ ]:
clean_df[["recent_form_difference"]].describe()

Findings

The `recent_form_difference` feature was successfully created by comparing the total points earned by both teams in their previous four matches.

Positive values indicate that the home team entered the match with better recent form, while negative values indicate better recent form for the away team.

Some missing values are expected because teams have not yet played four matches at the beginning of each season.

This feature will support Research Question 2 by allowing us to investigate whether teams in better recent form are more likely to achieve better match results.

In [ ]:
clean_df["recent_form_difference"].isna().sum()

Missing values are expected because, during the first matches of each season, teams have not yet played four previous matches. Therefore, recent form cannot be calculated for those observations.

### 3.9. Average Team Rating and Rating Difference

As we need TEAM players ratings and not individual ratings, for comparing between teams, we are calculating each TEAM (home and away) ratings and calculating the difference between them, in that case the positive rating will show a better rating for Home team and negative rating will show better rating for away team

`average_team_rating`

Our research question focuses on **team player ratings**, not individual player ratings.

To compare both teams fairly, we calculate the average rating of the home team and the away team using their defense, midfield, and attack ratings.

Finally, we create a new feature called `rating_difference`.

- A **positive** value indicates that the home team has a higher average rating.
- A **negative** value indicates that the away team has a higher average rating.
- A value close to **zero** indicates that both teams have similar overall ratings.

In [ ]:
clean_df["average_home_team_rating"] = clean_df[["home_attack_rating", "home_midfield_rating", "home_defense_rating"]].mean(axis=1) #axis 0 columns, 1 rows

In [ ]:
clean_df["average_away_team_rating"] = clean_df[["away_attack_rating", "away_midfield_rating", "away_defense_rating"]].mean(axis=1) #axis 0 columns, 1 rows

In [ ]:
clean_df["rating_difference"] = clean_df["average_home_team_rating"] - clean_df["average_away_team_rating"]

In [ ]:
clean_df[["average_home_team_rating", "average_away_team_rating", "rating_difference"]].head(5)

In [ ]:
clean_df[["average_home_team_rating","average_away_team_rating","rating_difference"]].describe()

Findings

The average team ratings were successfully calculated using the defense, midfield, and attack ratings.

The new `rating_difference` feature compares the overall quality of both teams before each match.

Positive values indicate a stronger home team, negative values indicate a stronger away team, while values close to zero suggest both teams have similar overall ratings.

This feature will support Research Question 4 by allowing us to investigate whether stronger team ratings are associated with better match performance.

clean_df.to_csv("cleaned_data.csv", index=False)

In [ ]:
clean_df.to_csv("cleaned_football_data.csv", index=False)

In [ ]:
cleaned_df = pd.read_csv("cleaned_football_data.csv")

cleaned_df.shape

In [ ]:
clean_df.shape

In [ ]:
cleaned_df.head()

# <span style="color:gold">Research Questions</span>

## 1. How accurately do Expected Goals (xG) predict the final match result?

@Berni

In [ ]:
import sqlite3
import pandas as pd

# Crea/conecta a una base de datos (se guarda como archivo .db en tu carpeta)
conn = sqlite3.connect("football.db")

# Vuelca clean_df como tabla SQL llamada "matches"
clean_df.to_sql("matches", conn, if_exists="replace", index=False)

In [ ]:
check = pd.read_sql_query("SELECT * FROM matches LIMIT 5", conn)
check

Calculate average goals per league

In [ ]:
query1 = """
SELECT league, AVG(total_goals) AS avg_goals
FROM matches
GROUP BY league
ORDER BY avg_goals DESC
"""
result1 = pd.read_sql_query(query1, conn)
result1

Average of total xgoals per season

In [ ]:
query2 = """
SELECT season, AVG(total_xg) AS avg_xg
FROM matches
GROUP BY season
ORDER BY season
"""
result2 = pd.read_sql_query(query2, conn)
result2

xG promedio vs goles reales promedio, por liga y temporada

In [ ]:
query3 = """
SELECT league, season,
       AVG(total_xg) AS avg_xg,
       AVG(total_goals) AS avg_goals,
       AVG(total_goals) - AVG(total_xg) AS avg_diff
FROM matches
GROUP BY league, season
ORDER BY league, season
"""
result3 = pd.read_sql_query(query3, conn)
result3

In [ ]:
correlation = clean_df["total_xg"].corr(clean_df["total_goals"])
print(f"Correlation between total_xg and total_goals: {correlation:.4f}")

In [ ]:
import numpy as np

# Create goal_difference
clean_df["goal_difference"] = clean_df["home_score"] - clean_df["away_score"]

# Create match_result
conditions = [
    clean_df["home_score"] > clean_df["away_score"],
    clean_df["home_score"] < clean_df["away_score"]
]
choices = ["Home Win", "Away Win"]
clean_df["match_result"] = np.select(conditions, choices, default="Draw")

In [ ]:
print(clean_df.columns.tolist())

In [ ]:
clean_df.to_sql("matches", conn, if_exists="replace", index=False)

In [ ]:
query4 = """
SELECT match_result, AVG(total_xg) AS avg_xg, COUNT(*) AS num_matches
FROM matches
GROUP BY match_result
"""
result4 = pd.read_sql_query(query4, conn)
result4

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograma - distribución de total_goals 
plt.figure(figsize=(8,5))
sns.histplot(clean_df["total_goals"], bins=15, kde=True)
plt.title("Distribución de Total Goals")
plt.show()

# Histograma - distribución de total_xg
plt.figure(figsize=(8,5))
sns.histplot(clean_df["total_xg"], bins=15, kde=True)
plt.title("Distribución de Total xG")
plt.show()

# Scatter plot: total_xg vs total_goals
plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x="total_xg", y="total_goals", alpha=0.5)
plt.title("Total xG vs Total Goals")
plt.show()

# Boxplot por liga
plt.figure(figsize=(10,6))
sns.boxplot(data=clean_df, x="league", y="total_goals")
plt.xticks(rotation=45)
plt.title("Total Goals by League")
plt.show()

In [ ]:
# Create xg_difference
clean_df["xg_difference"] = clean_df["home_xg"] - clean_df["away_xg"]

In [ ]:
clean_df.to_sql("matches", conn, if_exists="replace", index=False)

In [ ]:
correlation_diff = clean_df["xg_difference"].corr(clean_df["goal_difference"])
print(f"Correlation between xg_difference and goal_difference: {correlation_diff:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(data=clean_df, x="xg_difference", y="goal_difference", alpha=0.4)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.title("xG Difference vs Goal Difference")
plt.xlabel("xG Difference (home - away)")
plt.ylabel("Goal Difference (home - away)")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(clean_df["goal_difference"], bins=15, kde=True)
plt.title("Distribución de Goal Difference")
plt.show()

## RQ1: Is Expected Goals (xG) a good predictor of actual goals?

Expected Goals shows a moderate-to-strong relationship with actual goals scored. The correlation 
between `total_xg` and `total_goals` is **0.57**, meaning xG captures a good portion of scoring 
output, though finishing quality and luck still play a role match-to-match.

The relationship is even stronger when looking at match dominance: `xg_difference` and 
`goal_difference` correlate at **0.68**, suggesting that when a team creates significantly better 
chances than its opponent, that advantage is fairly likely to show up on the scoreboard.

**Conclusion:** xG is a reasonably good predictor of actual goals, particularly for identifying 
which team dominates a match — but it doesn't fully replace actual results, since finishing 
efficiency still introduces meaningful variance.

## 2. Does recent team performance influence match outcomes?

more Univariate Analysis

16332 rows - games from european leagues (bl, La Liga etc) Q3 2014 - Q2 2023 

* Relevant variables:

(from kaggle.com description of dataset)
- 11-12 - current standings (table positions) home/away team just before the start of the match
- (not 15-16 - total number of points home/away team have gained in their previous 4 matches)
- 19-20 - total number of goals home/away team have scored in their previous 4 matches 
- 23-24 - total number of goals home/away team have conceded (goals against) in their previous 4 matches
- (27-30 - average value of xG home/away team have gained in their previous 4 matches)

(each) - can be NaN in case of the first matches of the season when teams have not played enough matches this season yet

In [ ]:
"""
#bins for home_score - optional
import seaborn as sns
plt.figure(figsize=(8,6))
sns.histplot(
    data=clean_df,
    x='home_score',
    #hue='match_result',
    bins=20,
    palette={'Home Win':'skyblue','Draw':'lightgreen','Away Win':'salmon'},
    multiple='stack',
    edgecolor='black'
)
plt.xlabel('Home Goals')
plt.ylabel('Count')
plt.title('Distribution of Home Goals')
plt.tight_layout()


"""
clean_df["home_score"].value_counts()
clean_df["away_score"].value_counts()

In [ ]:
#clean_df['goal_difference'].value_counts().plot(kind='bar', color=['coral', 'gold'], edgecolor='white')

In [ ]:
clean_df["match_result"].value_counts()

total_matches = clean_df.shape[0]

clean_df["match_result"].value_counts()['Home Win'] / total_matches, clean_df["match_result"].value_counts()['Draw'] / total_matches, clean_df["match_result"].value_counts()['Away Win'] / total_matches

The Propability of winning is 0.45 for Home Team.

The Probability of winning is 0.25 for Away Team.

The Probability of a Draw is 0.3.

In [ ]:
import numpy as np

# Definiere die Bedingungen
conditions = [
    clean_df["recent_form_difference"] > 0,
    clean_df["recent_form_difference"] < 0,
    clean_df["recent_form_difference"] == 0
]

# Definiere die dazugehörigen Ausgaben
choices = ["Home Win", "Away Win", "Draw"]

# Wende np.select an, um die neue Spalte zu erstellen
clean_df["recent_form_difference_ten"] = np.select(conditions, choices, default="")

# Kurze Kontrolle, ob es geklappt hat:
print(clean_df[["recent_form_difference", "recent_form_difference_ten"]].tail())

In [ ]:
pd.crosstab(clean_df["recent_form_difference_ten"] ,clean_df["match_result"] )

The recent form predicts 

a Home Win with certainty

3719 / (1492 + 1652 + 3719) = 0.54 <span style="color:red">!</span>  #not bad

an Away Win with certainty

2872 / (2872+ 1895 + 2653) = 0.38

and a Draw with certainty

410 / (471 + 410 + 739) = 0.25

vs. Propabilitys in dataset:

The Propability of winning is <span style="color:red">0.45</span> for Home Team.

The Probability of winning is 0.25 for Away Team.

The Probability of a Draw is 0.3.

nb: agent's opinion:
RQ2 (Recent Form vs Outcome): Showed a clear increase in home win probability (54.19% vs <span style="color:orange">35.75%</span>) when the home team is in better recent form.

# 3. Do teams with more current points consistently achieve better match results?

Objective: Investigate whether teams with a higher number of league points before a match are
more likely to obtain better match results. 
Expected Goals (xG) will be used as a supporting variable rather than the main explanatory variable

## Initial Hypothesis

We expect teams with more league points before a match to achieve better match results more frequently.

However, football is unpredictable, therefore we also expect to find matches where teams with fewer points still achieve positive results.

Expected Goals (xG) will be analysed as a supporting metric to determine whether it reinforces or contradicts the relationship between current points and match outcomes.

## Variables Used

To answer this research question, we will use both original dataset variables and engineered features.

### Primary variables
- home_points
- away_points
- points_difference
- match_result

These variables directly measure the teams' league performance before each match.

### Supporting variables
- home_xg
- away_xg
- xg_difference
- goal_difference
- league
- season

These variables provide additional context by comparing expected performance, actual goal difference and possible differences across leagues and seasons.

In [ ]:
clean_df.home_points.head()


In [ ]:
plt.figure(figsize=(8,4))

sns.histplot(data=clean_df, x= "home_points")
plt.title("Distribution of Home Team Points")
plt.xlabel(" Home Team Points")
plt.ylabel("Number of Matches")
plt.show()


The distribution shows that home teams cover the full range of league standings, although most matches involve teams with relatively low or medium point totals. Several outliers represent teams with exceptionally high league points.

In [ ]:
clean_df.away_points.head()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=clean_df, x= "away_points")
plt.title("Distribution of Away Team Points")
plt.xlabel("Away Team Points")
plt.ylabel("Number of Matches")
plt.show()

In [ ]:
clean_df.points_difference.head()

Home and Away points present some similiar distribution.

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=clean_df, x="points_difference")
plt.title("Distribution of Points Between Home and Away Teams")
plt.xlabel("Points Difference")
plt.ylabel("Number of Matches")
plt.show()

In [ ]:
plt.figure(figsize=(8, 2.5))
sns.boxplot(data=clean_df, x="points_difference")
plt.title("Distribution of Points Difference")
plt.xlabel("Points Difference")
plt.show()

In [ ]:
clean_df.points_difference.describe()

We expected the distribution to be centered around zero because many matches involve teams with similar league standings, while large point differences should be less frequent.

The distribution of points_difference is centered around zero, suggesting that many matches involve teams with similar league standings.

Both positive and negative tails indicate that some matches are played between teams with very different numbers of accumulated points.

The boxplot confirms that most matches involve teams with relatively small differences in league points.

Several outliers appear on both sides of the distribution, representing matches between teams with very different league standings.

These outliers are expected in football competitions and correspond to real competitive differences rather than data quality issues.

### Points Diferrence (numerical) VS Match Result (categorical)

 Relationship Between Points Difference and Match Result

This analysis evaluates whether differences in league points before a match are associated with the final match outcome.

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=clean_df, x="match_result", y="points_difference", order=["Home Win", "Draw", "Away Win"])
plt.title("League Points Difference by Match Result")
plt.xlabel("Match Result")
plt.ylabel("League Points Difference")
plt.show()

In [ ]:
clean_df.groupby("match_result")["points_difference"].describe().round(2)

Home victories generally occur when the home team has a positive league points advantage.

Away victories are more common when the away team has accumulated more league points.

Draws tend to occur between teams with relatively similar league standings.

Although many outliers exist, the overall pattern suggests that current league points are associated with match outcomes.

### Points Difference vs Goal Difference

Relationship Between League Points Difference and Goal Difference

Teams with more league points also tend to generate higher expected goals.

This suggests that stronger teams not only collect more points but also create better scoring opportunities.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=clean_df, x="points_difference", y="goal_difference",alpha=0.35)
plt.title("League Points Difference vs Goal Difference")
plt.xlabel("League Points Difference")
plt.ylabel("Goal Difference")
plt.show()

In [ ]:
clean_df[["points_difference","goal_difference"]].corr()

A positive relationship is observed between league points difference and goal difference.

Teams with larger advantages in league points generally achieve larger winning margins, although considerable variability remains.

### Points Difference vs Expected Goals Difference

Points Difference vs Expected Goals Difference

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=clean_df, x="points_difference", y="xg_difference", alpha=0.35)
plt.title("League Points Difference vs Expected Goals Difference")
plt.xlabel("League Points Difference")
plt.ylabel("Expected Goals Difference")
plt.show()

In [ ]:
clean_df[["points_difference","xg_difference"]].corr()

### SQL Analysis Validation

To validate the findings obtained during the exploratory data analysis in Pandas, we perform several SQL queries on the original database.

These queries allow us to confirm whether the observed relationships remain consistent when analysed directly from the relational database.

In [ ]:
import sqlite3
conn = sqlite3.connect("football_project.db")
clean_df.to_sql("cleaned_matches", conn, if_exists="replace", index=False)

 SQL Analysis 1

Average league points difference by match result.

In [ ]:
query = """
SELECT
    match_result,
    ROUND(AVG(points_difference),2) AS avg_points_difference
FROM cleaned_matches
GROUP BY match_result;
"""

pd.read_sql(query, conn)

The SQL results confirm the trend observed during the EDA.

Matches won by the home team generally present positive league point differences, while away victories are associated with negative point differences.

SQL Analysis 2

Average league points difference by league.

In [ ]:
query = """
SELECT
    league,
    ROUND(AVG(points_difference),2) AS avg_points_difference
FROM cleaned_matches
GROUP BY league;
"""

pd.read_sql(query, conn)

findings?

SQL Analysis 3

Average league points difference by season.

In [ ]:
query = """
SELECT
    season,
    ROUND(AVG(points_difference),2) AS avg_points_difference
FROM cleaned_matches
GROUP BY season
ORDER BY season;
"""

pd.read_sql(query, conn)

### Aditional Tables

Estatistics By League

In [ ]:
clean_df.groupby("league")["points_difference"].describe().round(2)

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=clean_df, x="league", y="points_difference")
plt.title("League Points Difference by League")
plt.xlabel("League")
plt.ylabel("League Points Difference")
plt.show()

Estatistics By Seasons

In [ ]:
clean_df.groupby("season")["points_difference"].describe().round(2)

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=clean_df, x="season", y="points_difference")
plt.title("League Points Difference by Season")
plt.xlabel("Season")
plt.ylabel("League Points Difference")
plt.xticks(rotation=45)
plt.show()

### Findings

Teams with higher league points before a match generally achieve better match results.

Positive league points differences are associated with home victories, while negative differences are more common in away victories.

Teams with larger league point advantages also tend to produce higher goal differences and higher Expected Goals (xG).

However, football remains unpredictable, as numerous exceptions and outliers demonstrate that league points alone cannot fully explain match outcomes.

## Conclusion

The analysis suggests that current league points are a useful indicator of future match performance.

Although league points do not perfectly predict results, teams with higher accumulated points generally obtain better outcomes and generate stronger performances.

Therefore, current league points can be considered an informative predictor of match performance, especially when combined with complementary variables such as Expected Goals (xG).

---



----

## 4. How do team player ratings and team formations influence match performance?

### 1. Univariate Analysis

We first explore individual distributions of key variables: rating_difference and team formations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(clean_df['rating_difference'].dropna(), bins=40)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='equal rating')
axes[0].set_title('Distribution of Rating Difference')
axes[0].set_xlabel('Rating Difference')
axes[0].set_ylabel('Number of Matches')
axes[0].legend()

axes[1].boxplot(clean_df['rating_difference'].dropna(), vert=True)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Boxplot of Rating Difference')
axes[1].set_ylabel('Rating Difference')

plt.tight_layout()
plt.show()

print(clean_df['rating_difference'].describe().round(3))

**Findings:**

- The `rating_difference` distribution is roughly symmetric and centered near zero — in most matches both teams have similar overall ratings.
- The spread covers approximately from -5 to +5, with very few extreme outliers.
- The red dashed line at 0 marks equal ratings; a substantial proportion of matches fall near this line.

#### Distribution of Formations

We examine which formations are most commonly used by home and away teams.

> **Formation Encoding Key** (from the dataset documentation):
> - **0** → 2 defenders, 1 forward (e.g., 4-2-3-1, 4-3-3, 4-5-1)
> - **1** → 3 defenders, 1 forward (e.g., 3-4-2-1, 3-5-1-1, 5-2-2-1)
> - **2** → 2 defenders, 2 forwards (e.g., 4-4-2, 4-2-2-2)
> - **3** → 3 defenders, 2 forwards (e.g., 3-5-2, 3-4-1-2, 5-3-2)
> - **4** → Other / unusual formations (anything not fitting the above patterns)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

home_form = clean_df['home_formation'].value_counts().head(10)
axes[0].bar(home_form.index.astype(str), home_form.values, color='steelblue', edgecolor='white')
axes[0].set_title('Top 10 Home Team Formations')
axes[0].set_xlabel('Formation')
axes[0].set_ylabel('Number of Matches')
axes[0].tick_params(axis='x', rotation=45)

away_form = clean_df['away_formation'].value_counts().head(10)
axes[1].bar(away_form.index.astype(str), away_form.values, color='coral', edgecolor='white')
axes[1].set_title('Top 10 Away Team Formations')
axes[1].set_xlabel('Formation')
axes[1].set_ylabel('Number of Matches')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Top 5 Home Formations:")
print(clean_df['home_formation'].value_counts().head(5))
print("\nTop 5 Away Formations:")
print(clean_df['away_formation'].value_counts().head(5))

**Findings:**

- The most common formation for both home and away teams is **formation 0** (4-2-3-1 / 4-3-3), followed by formation 2 (4-4-2).
- Home teams tend to use slightly more attacking formations.
- The distribution of formations is similar between home and away teams, suggesting formations alone may not explain home advantage.

### 2. Bivariate Analysis

Now we explore relationships between two variables at a time.

rating_difference vs goal_difference

We investigate whether a higher team rating advantage translates into a larger goal margin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(clean_df['rating_difference'], clean_df['goal_difference'], alpha=0.3, c='steelblue', s=12, edgecolors='none')
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)

from numpy.polynomial.polynomial import polyfit
valid = clean_df[['rating_difference', 'goal_difference']].dropna()
x_fit = valid['rating_difference']
y_fit = valid['goal_difference']
b, m = polyfit(x_fit, y_fit, 1)
x_line = np.linspace(x_fit.min(), x_fit.max(), 100)
axes[0].plot(x_line, b + m * x_line, color='darkred', linewidth=2, label=f'trend (slope={m:.2f})')

axes[0].set_title('Rating difference vs Goal difference')
axes[0].set_xlabel('Rating difference (Home − Away)')
axes[0].set_ylabel('Goal difference (Home − Away)')
axes[0].legend()

hb = axes[1].hexbin(clean_df['rating_difference'], clean_df['goal_difference'], gridsize=40, cmap='YlOrRd', mincnt=1)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].axvline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Rating difference vs Goal difference (density)')
axes[1].set_xlabel('Rating difference (Home − Away)')
axes[1].set_ylabel('Goal difference (Home − Away)')
plt.colorbar(hb, ax=axes[1], label='Number of matches')

plt.tight_layout()
plt.show()

corr_rating_goal_diff = clean_df['rating_difference'].corr(clean_df['goal_difference'])
print(f"Correlation (rating_difference vs goal_difference): {corr_rating_goal_diff:.3f}")

**Findings:**

- There is a **positive relationship** between `rating_difference` and `goal_difference`: teams with higher ratings tend to outscore their opponents.
- The hexbin plot shows the highest density of matches is near (0, 0) — when ratings are similar, goal differences are small.
- The trend line confirms that as rating advantage increases, the expected goal margin grows.

#### `rating_difference` vs `match_result`

We check whether the rating difference is associated with the final match outcome.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=clean_df, x='match_result', y='rating_difference', order=['Away Win', 'Draw', 'Home Win'])
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title('Rating Difference by Match Result')
ax.set_xlabel('Match Result')
ax.set_ylabel('Rating Difference (Home \u2212 Away)')

plt.tight_layout()
plt.show()

print("Average rating_difference by match_result:")
print(clean_df.groupby('match_result')['rating_difference'].describe().round(3))

**Findings:**

- **Home Wins** are associated with a **positive** median `rating_difference` — the home team is typically rated higher.
- **Away Wins** show a **negative** median `rating_difference` — the away team is rated higher.
- **Draws** cluster around zero, confirming that evenly matched teams are more likely to draw.
- This pattern strongly supports the hypothesis that player ratings are predictive of match outcomes.

#### Formation vs Match Result

We analyze whether certain formations are linked to winning more often (home team perspective).

In [ ]:
# Cross-tab: home formation vs match result (top 8 formations)
top_formations = clean_df['home_formation'].value_counts().head(8).index
filtered = clean_df[clean_df['home_formation'].isin(top_formations)]

ct = pd.crosstab(filtered['home_formation'], filtered['match_result'])

# Calculate win % for each formation
ct['Total'] = ct.sum(axis=1)
ct['Home_Win_Pct'] = (ct['Home Win'] / ct['Total'] * 100).round(1)
ct['Away_Win_Pct'] = (ct['Away Win'] / ct['Total'] * 100).round(1)
ct['Draw_Pct'] = (ct['Draw'] / ct['Total'] * 100).round(1)

# Sort by Home Win %
ct_sorted = ct.sort_values('Home_Win_Pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(ct_sorted))
width = 0.25

ax.bar([i - width for i in x], ct_sorted['Home_Win_Pct'], width, label='Home Win %', color='steelblue', edgecolor='white')
ax.bar(x, ct_sorted['Draw_Pct'], width, label='Draw %', color='gold', edgecolor='white')
ax.bar([i + width for i in x], ct_sorted['Away_Win_Pct'], width, label='Away Win %', color='coral', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(ct_sorted.index.astype(str), rotation=45)
ax.set_title('Match Result Percentage by Home Formation (Top 8)')
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('Home Formation')
ax.legend()
ax.axhline(color='steelblue', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("Match Result by Home Formation (Top 8):")
ct_sorted[['Home_Win_Pct', 'Draw_Pct', 'Away_Win_Pct', 'Total']]

**Findings:**

- While there is some variation in win percentages across formations, the differences are relatively small.
- No single formation guarantees a dramatically higher win rate — the baseline home win rate (~45%) is broadly consistent across all common formations.
- This suggests that **formation alone is not a strong predictor of match outcome**; team quality (ratings) matters more.


### 4. Conclusions for Q4

#### Key Takeaways

1. **Player ratings strongly influence match outcomes.**
   - The correlation between `rating_difference` and `goal_difference` confirms that teams with higher-rated players tend to score more and concede fewer goals.
   - The boxplot of `rating_difference` by `match_result` shows a clear gradient: home wins when the home team has a rating advantage, away wins when the away team is stronger, and draws when ratings are balanced.

2. **Formations have a limited standalone impact.**
   - While certain formations show slightly higher win percentages, the differences are modest. No formation consistently outperforms the baseline home win rate (~45%) by a large margin.
   - The most popular formations (0: 4-2-3-1/4-3-3, 2: 4-4-2) are used by both strong and weak teams, so formation alone does not determine success.

3. **League-level differences exist but are small.**
   - The SQL analysis reveals that the top European leagues feature similar average team ratings, with only narrow spreads.
   - The average `rating_difference` across leagues is consistently near zero, meaning home and away teams are generally well-matched.

4. **Integration with earlier findings.**
   - Combined with Q1\u2013Q3, we see that **ratings complement xG and recent form** as predictors: teams with higher ratings generate more xG, win more points, and have better recent form \u2014 all reinforcing each other.
   - Player ratings provide a *structural* measure of team quality, while xG and recent form capture *situational* performance.

#### Final words

> **Team player ratings are a strong predictor of match performance.** Higher-rated teams consistently score more goals, generate more expected goals, and win more matches. Team formation, by contrast, shows only a weak association with match outcomes and should be considered a secondary factor.